# AhorraMás Web Scraper

This notebook collects product data from AhorraMás using **Selenium and BeautifulSoup**.

### How it works

1. Opens the AhorraMás website and automatically accepts cookies.
2. Searches for each product term defined in `SEARCH_TERMS`.
3. Loads additional products by clicking the **"Más resultados"** button until no more results are available.
4. Extracts product names, prices, units, IDs, URLs, and collection timestamps.
5. Combines all search results and removes duplicate products.
6. Validates the collected data and reports incomplete searches.

### Data Storage

- **RAW:** `data/raw/ahorramas_products_raw.csv`
- **PROCESSED:** `data/processed/ahorramas_products.csv`
- **HISTORY:** `data/history/ahorramas/`

Historical files are saved by collection date (`YYYY-MM-DD`) to support price comparisons over time. A quality report records the status of each search.


**Note:** Some searches may return incomplete results due to website loading issues. These limitations are recorded and must be considered when analyzing price trends.

In [ ]:

import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin, quote

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.ahorramas.com/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# TEST MODE
# --------------------------------------------------

TEST_MODE = False
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

print("Search terms to scrape:", TERMS_TO_SCRAPE)


# --------------------------------------------------
# PROJECT DIRECTORIES
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError("Could not locate the project root.")


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
HISTORY_DIR = BASE_DIR / "data" / "history" / "ahorramas"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

# Current files used by the pipeline
RAW_PATH = RAW_DIR / "ahorramas_products_raw.csv"
PROCESSED_PATH = PROCESSED_DIR / "ahorramas_products.csv"

# Collection date
RUN_DATE = datetime.now().strftime("%Y-%m-%d")

# Historical snapshots
RAW_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_raw.csv"
PROCESSED_HISTORY_PATH = HISTORY_DIR / f"{RUN_DATE}_processed.csv"

print("Project:", BASE_DIR)
print("Raw:", RAW_PATH)
print("Processed:", PROCESSED_PATH)
print("History:", HISTORY_DIR)
print("Collection date:", RUN_DATE)


# --------------------------------------------------
# SELENIUM DRIVER CONFIGURATION
# --------------------------------------------------

def start_ahorramas_driver():
    options = Options()

    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "excludeSwitches", ["enable-automation"]
    )
    options.add_experimental_option(
        "useAutomationExtension", False
    )
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)

    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', "
        "{get: () => undefined})"
    )

    driver.get(HOME_URL)

    return driver


def accept_ahorramas_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir')]",
        "//button[@id='onetrust-accept-btn-handler']"
    ]

    for xpath in xpaths:
        try:
            buttons = driver.find_elements(By.XPATH, xpath)

            for btn in buttons:
                if btn.is_displayed():
                    btn.click()
                    print("AhorraMás cookies accepted.")
                    return

        except Exception:
            pass

    print("Cookie banner not found or already accepted.")


# --------------------------------------------------
# PRODUCT SEARCH AND PAGINATION
# --------------------------------------------------

def count_product_links(driver):
    # Count unique links matching potential product URLs
    links = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*=".html"], a[class*="product"], div.product-tile a'
    )

    hrefs = {
        link.get_attribute("href")
        for link in links
        if link.get_attribute("href")
    }

    return len(hrefs)


def search_ahorramas(driver, term):
    encoded_term = quote(term)
    search_url = f"https://www.ahorramas.com/search?q={encoded_term}"

    print(f"Navigating to: {search_url}")

    driver.get(search_url)
    time.sleep(4)


def load_all_ahorramas_products(driver, max_clicks=100):

    print("Loading all available products...")

    clicks = 0
    selector = "button.more[data-url]"

    while clicks < max_clicks:

        # Find the "More results" button in the DOM
        buttons = driver.find_elements(By.CSS_SELECTOR, selector)

        enabled_buttons = [
            button for button in buttons
            if button.is_displayed() and button.is_enabled()
        ]

        if not enabled_buttons:

            # Distinguish hidden buttons from completed pagination
            if buttons:
                print("More results button exists but is not visible.")
                return count_product_links(driver), "button_not_visible"

            print("No more results button found.")
            return count_product_links(driver), "complete"

        button = enabled_buttons[0]

        # Scroll directly to the button instead of the page bottom
        driver.execute_script(
            "arguments[0].scrollIntoView({"
            "behavior: 'instant', block: 'center'"
            "});",
            button
        )

        time.sleep(0.5)

        previous_count = count_product_links(driver)
        previous_url = button.get_attribute("data-url")

        try:
            # Load the next batch of products
            button.click()

            # Wait until new products or updated pagination are detected
            WebDriverWait(driver, 20).until(
                lambda d: (
                    count_product_links(d) > previous_count
                    or not any(
                        b.is_displayed()
                        and b.get_attribute("data-url") == previous_url
                        for b in d.find_elements(
                            By.CSS_SELECTOR, selector
                        )
                    )
                )
            )

        except TimeoutException:
            print("Timeout loading additional products.")
            return count_product_links(driver), "timeout"

        except Exception as exc:
            print(f"Error: {type(exc).__name__}: {exc}")
            return count_product_links(driver), "error"

        clicks += 1

        current_count = count_product_links(driver)

        print(
            f"Click {clicks}: "
            f"{previous_count} -> {current_count} links"
        )

    print("Maximum number of clicks reached.")
    return count_product_links(driver), "max_clicks_reached"


# --------------------------------------------------
# PRODUCT DATA EXTRACTION
# --------------------------------------------------

def extract_ahorramas_products(driver, searched_item):

    soup = BeautifulSoup(driver.page_source, "html.parser")

    cards = soup.select(
        'div.product-tile, li.grid-tile, div.grid-tile, '
        'div[class*="product-tile"]'
    )

    if not cards:
        cards = soup.select('div[class*="product"], article')

    records = []
    seen_urls = set()

    scraped_at = datetime.now().isoformat(timespec="seconds")

    for card in cards:
        text = card.get_text(" ", strip=True)

        # Identify the product URL
        link_elem = card.find(
            "a", href=re.compile(r"\.html|/p/")
        )

        if not link_elem:
            link_elem = card.find("a", href=True)

        if not link_elem:
            continue

        href = link_elem.get("href")

        if not href or "search" in href or "cart" in href:
            continue

        url = urljoin(HOME_URL, href)

        if url in seen_urls:
            continue

        seen_urls.add(url)

        # Extract product name
        name = ""

        for tag in ["h3", "h2", "h4", "a", "span", "div"]:
            elem = card.find(
                tag,
                class_=re.compile("title|name|desc|link", re.I)
            )

            if elem:
                txt = elem.get_text(" ", strip=True)

                if len(txt) > 4 and "€" not in txt:
                    name = txt
                    break

        if not name:
            link_text = link_elem.get_text(" ", strip=True)

            if len(link_text) > 4:
                name = link_text

        if not name:
            continue

        # Extract total product price
        price = None
        price_raw = None

        price_match = re.search(
            r"(\d+(?:[.,]\d{1,2})?)\s*€",
            text
        )

        if price_match:
            price_raw = price_match.group(0)
            price = float(
                price_match.group(1).replace(",", ".")
            )

        # Discard only missing, zero or negative prices
        if price is None or price <= 0:
            continue

        # Extract product ID from URL
        id_match = re.search(
            r"/(\d+)\.html|/p/(\d+)", url
        )

        product_id = (
            id_match.group(1) or id_match.group(2)
            if id_match else None
        )

        # --------------------------------------------------
        # EXTRACT QUANTITY AND UNIT PRICE
        # --------------------------------------------------

        unit = None
        price_per_unit = None
        quantity_val = None

        name_lower = name.lower()

        # Detect liters and milliliters
        l_match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(?:l|litros?)\b",
            name_lower
        )

        ml_match = re.search(
            r"(\d+)\s*ml\b",
            name_lower
        )

        # Detect kilograms and grams
        kg_match = re.search(
            r"(\d+(?:[.,]\d+)?)\s*(?:kg|k)\b",
            name_lower
        )

        g_match = re.search(
            r"(\d+)\s*(?:g|gr)\b",
            name_lower
        )

        # Detect individual units
        u_match = re.search(
            r"(\d+)\s*(?:u|uds|unidades?)\b",
            name_lower
        )

        if l_match:
            quantity_val = float(
                l_match.group(1).replace(",", ".")
            )
            unit = "LITRO"

        elif ml_match:
            quantity_val = float(ml_match.group(1)) / 1000.0
            unit = "LITRO"

        elif kg_match:
            quantity_val = float(
                kg_match.group(1).replace(",", ".")
            )
            unit = "KILO"

        elif g_match:
            quantity_val = float(g_match.group(1)) / 1000.0
            unit = "KILO"

        elif u_match:
            quantity_val = float(u_match.group(1))
            unit = "UNIDAD"

            if quantity_val == 12 and "docena" in name_lower:
                unit = "DOCENA"

        # Calculate price per unit when quantity is available
        if quantity_val and quantity_val > 0:
            price_per_unit = round(
                price / quantity_val, 4
            )

        # Store extracted product information
        records.append({
            "searched_item": searched_item,
            "supermarket": "AhorraMás",
            "product_id": product_id,
            "name": name,
            "quantity_raw": None,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": None,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)

    print(
        f"Products extracted for '{searched_item}': {len(df)}"
    )

    return df


Búsquedas que se ejecutarán: ['leche', 'huevos', 'manzana', 'plátano', 'tomate', 'arroz', 'pasta', 'pan', 'pollo', 'carne picada', 'salmón', 'yogur', 'queso', 'agua', 'zumo', 'atún', 'garbanzos', 'aceite', 'detergente', 'papel higiénico']
Proyecto: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Raw: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw\ahorramas_products_raw.csv
Processed: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\processed\ahorramas_products.csv
History: C:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\history\ahorramas
Collection date: 2026-11-10


In [12]:
all_results = []
summary = []

driver = start_ahorramas_driver()

try:
    time.sleep(3)
    accept_ahorramas_cookies(driver)

    for i, term in enumerate(TERMS_TO_SCRAPE, start=1):
        print("\n" + "=" * 70)
        print(f"[{i}/{len(TERMS_TO_SCRAPE)}] Buscando en AhorraMás: {term}")

        start = time.monotonic()

        try:
            search_ahorramas(driver, term)
            loaded_count, load_status = load_all_ahorramas_products(driver)
            df_term = extract_ahorramas_products(driver, term)

            elapsed = round(time.monotonic() - start, 2)

            if not df_term.empty:
                all_results.append(df_term)

            summary.append({
                "searched_item": term,
                "status": load_status,
                "products_loaded": loaded_count,
                "rows_extracted": len(df_term),
                "seconds": elapsed,
            })

            print(f"✓ {len(df_term)} filas extraídas en {elapsed} s")
            time.sleep(2)

        except Exception as exc:
            elapsed = round(time.monotonic() - start, 2)
            print(f"ERROR con '{term}': {type(exc).__name__}: {exc}")
            summary.append({
                "searched_item": term,
                "status": "error",
                "products_loaded": None,
                "rows_extracted": 0,
                "seconds": elapsed,
            })
            break

finally:
    driver.quit()

df_ahorramas = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
df_summary = pd.DataFrame(summary)
display(df_summary)
print("\nFilas raw totales:", len(df_ahorramas))

Cookies aceptadas en AhorraMás.

[1/20] Buscando en AhorraMás: leche
Navegando a: https://www.ahorramas.com/search?q=leche
Cargando todos los productos...
Click 1: 43 -> 62 links
Click 2: 62 -> 82 links
Click 3: 82 -> 102 links
Click 4: 102 -> 122 links
Click 5: 122 -> 142 links
Click 6: 142 -> 162 links
Click 7: 162 -> 182 links
Click 8: 182 -> 193 links
No more results button found.
Productos extraídos para 'leche': 171
✓ 171 filas extraídas en 140.76 s

[2/20] Buscando en AhorraMás: huevos
Navegando a: https://www.ahorramas.com/search?q=huevos
Cargando todos los productos...
No more results button found.
Productos extraídos para 'huevos': 20
✓ 20 filas extraídas en 7.14 s

[3/20] Buscando en AhorraMás: manzana
Navegando a: https://www.ahorramas.com/search?q=manzana
Cargando todos los productos...
Click 1: 55 -> 70 links
No more results button found.
Productos extraídos para 'manzana': 38
✓ 38 filas extraídas en 15.08 s

[4/20] Buscando en AhorraMás: plátano
Navegando a: https://www.

,searched_item,status,products_loaded,rows_extracted,seconds
0,leche,complete,193,171,140.76
1,huevos,complete,42,20,7.14
2,manzana,complete,70,38,15.08
3,plátano,complete,59,32,17.70
4,tomate,complete,87,59,36.81
5,arroz,error,42,20,10.17
6,pasta,error,42,20,10.14
7,pan,complete,179,157,112.64
8,pollo,complete,137,99,57.20
9,carne picada,error,41,19,10.72



Filas raw totales: 1412


In [13]:
# --------------------------------------------------
# PREPARE RAW DATA
# --------------------------------------------------

if df_ahorramas.empty:
    print("No raw data collected.")
else:
    print("Raw data ready for processing.")
    print("Total rows:", len(df_ahorramas))
    print("Search terms:", df_ahorramas["searched_item"].nunique())


Raw data ready for processing.
Total rows: 1412
Search terms: 20


In [15]:
# --------------------------------------------------
# PROCESS DATA
# --------------------------------------------------

if df_ahorramas.empty:
    raise RuntimeError("No products collected.")

df_clean = df_ahorramas.copy()

# Deduplicate products
if (
    "product_id" in df_clean.columns
    and df_clean["product_id"].notna().any()
):
    with_id = (
        df_clean[df_clean["product_id"].notna()]
        .drop_duplicates(subset=["product_id"])
    )

    without_id = (
        df_clean[df_clean["product_id"].isna()]
        .drop_duplicates(subset=["name", "price"])
    )

    df_clean = pd.concat(
        [with_id, without_id], ignore_index=True
    ).reset_index(drop=True)

else:
    df_clean = (
        df_clean.drop_duplicates(subset=["name", "price"])
        .reset_index(drop=True)
    )

print("Raw rows:", len(df_ahorramas))
print("Processed products:", len(df_clean))


# --------------------------------------------------
# VALIDATE COLLECTION
# --------------------------------------------------

if TEST_MODE:
    raise RuntimeError(
        "Test mode enabled. Production files not saved."
    )

if len(df_summary) != len(SEARCH_TERMS):
    raise RuntimeError(
        "Not all search terms were executed."
    )

# Report incomplete searches without stopping the collection

incomplete_searches = df_summary[
    df_summary["status"] != "complete"
]

if not incomplete_searches.empty:
    print(
        f"WARNING: {len(incomplete_searches)} searches "
        "did not finish completely."
    )

    display(incomplete_searches[
        ["searched_item", "status", "products_loaded", "rows_extracted"]
    ])

if df_summary["rows_extracted"].le(0).any():
    raise RuntimeError(
        "Some searches returned no products."
    )

if df_clean.empty:
    raise RuntimeError(
        "Processed dataset is empty."
    )

if (
    df_clean["price"].isna().any()
    or df_clean["price"].le(0).any()
):
    raise RuntimeError(
        "Missing or invalid prices detected."
    )


# --------------------------------------------------
# CHECK HISTORICAL FILES
# --------------------------------------------------

if RAW_HISTORY_PATH.exists() or PROCESSED_HISTORY_PATH.exists():
    raise FileExistsError(
        f"Historical files already exist for {RUN_DATE}."
    )


# --------------------------------------------------
# SAVE HISTORICAL SNAPSHOTS
# --------------------------------------------------

df_ahorramas.to_csv(
    RAW_HISTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)

df_clean.to_csv(
    PROCESSED_HISTORY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Historical files saved.")


# --------------------------------------------------
# UPDATE CURRENT DATA
# --------------------------------------------------

df_ahorramas.to_csv(
    RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

df_clean.to_csv(
    PROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Current files updated.")
print("Collection date:", RUN_DATE)
print("Final products:", len(df_clean))

display(df_clean.head(10))


Raw rows: 1412
Processed products: 1347


,searched_item,status,products_loaded,rows_extracted
5,arroz,error,42,20
6,pasta,error,42,20
9,carne picada,error,41,19


Historical files saved.
Current files updated.
Collection date: 2026-11-10
Final products: 1347


,searched_item,supermarket,product_id,name,quantity_raw,price_raw,price,price_per_unit_raw,price_per_unit,unit,url,scraped_at
0,leche,AhorraMás,None,Leche semidesnatada Alipende 1L,None,"0,83 €",0.83,None,0.8300,LITRO,https://www.ahorramas.com/leche-semidesnatada-...,2026-11-10T12:07:20
1,leche,AhorraMás,None,Leche Lauki 1l semidesnatada,None,"1,09 €",1.09,None,1.0900,LITRO,https://www.ahorramas.com/leche-lauki-1l-semid...,2026-11-10T12:07:20
2,leche,AhorraMás,None,Leche entera Alipende 1L,None,"0,96 €",0.96,None,0.9600,LITRO,https://www.ahorramas.com/leche-entera-alipend...,2026-11-10T12:07:20
3,leche,AhorraMás,None,Leche desnatada Alipende 1L,None,"0,81 €",0.81,None,0.8100,LITRO,https://www.ahorramas.com/leche-desnatada-alip...,2026-11-10T12:07:20
4,leche,AhorraMás,None,Leche Lauki 1l entera,None,"1,18 €",1.18,None,1.1800,LITRO,https://www.ahorramas.com/leche-lauki-1l-enter...,2026-11-10T12:07:20
5,leche,AhorraMás,None,Leche Asturiana 1l semidesnatada,None,"1,17 €",1.17,None,1.1700,LITRO,https://www.ahorramas.com/leche-asturiana-1l-s...,2026-11-10T12:07:20
6,leche,AhorraMás,None,Leche Asturiana 1l entera,None,"1,29 €",1.29,None,1.2900,LITRO,https://www.ahorramas.com/leche-asturiana-1l-e...,2026-11-10T12:07:20
7,leche,AhorraMás,None,Leche Puleva calcio 1l desnatada,None,"1,59 €",1.59,None,1.5900,LITRO,https://www.ahorramas.com/leche-puleva-calcio-...,2026-11-10T12:07:20
8,leche,AhorraMás,None,Leche Río 1l entera,None,"1,14 €",1.14,None,1.1400,LITRO,https://www.ahorramas.com/leche-rio-1l-entera-...,2026-11-10T12:07:20
9,leche,AhorraMás,None,"Leche Asturiana 1,5l semidesnatada",None,"1,19 €",1.19,None,0.7933,LITRO,https://www.ahorramas.com/leche-asturiana-15l-...,2026-11-10T12:07:20
